# Progetto 40 — Transfer Learning con ResNet50 per la Smart Agriculture

Classificatore di immagini aeree da drone (AgriFuture) in 5 classi — Grano, Mais, Soia, Riso, Terreno vuoto — costruito su ResNet50 pre-addestrata su ImageNet, con dati simulati.

## 1. Setup

### 1.1 Import

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


### 1.2 Configurazione parametri

1. Input shape `(224, 224, 3)`, standard ottimale per ResNet50
2. 5 classi distinte
3. Numero di campioni simulati e `batch_size` (da ridurre se il computer è lento)

In [3]:
SEED = 21
tf.keras.utils.set_random_seed(SEED)

INPUT_SHAPE = (224, 224, 3)
CLASS_NAMES = ['Grano', 'Mais', 'Soia', 'Riso', 'Terreno vuoto']
NUM_CLASSES = len(CLASS_NAMES)
NUM_SAMPLES = 200
BATCH_SIZE = 32

#maiuscolo perché sono costanti

## 2. Generazione dei dati simulati

1. Funzione generatrice di immagini random e label, senza caricare file dal disco (script standalone)
2. Etichette `y` in One-Hot Encoding, coerenti con il numero di classi (5)

In [7]:
def genera_dati(num_samples, input_shape, num_classes):
    X = np.random.rand(num_samples, *input_shape).astype(np.float32)
    indici = np.random.randint(0, num_classes, num_samples)
    y = tf.keras.utils.to_categorical(indici, num_classes)
    return X, y

X_train, y_train = genera_dati(NUM_SAMPLES, INPUT_SHAPE, NUM_CLASSES)
print(X_train.shape, y_train.shape)

X_test, y_test = genera_dati(50, INPUT_SHAPE, NUM_CLASSES)


(200, 224, 224, 3) (200, 5)


## 3. Costruzione del modello (Transfer Learning)

1. Backbone: ResNet50 pre-addestrata su ImageNet (non MobileNetV2), importata senza il livello superiore (`include_top=False`)
2. Pesi della rete base congelati (`trainable = False`)
3. Head personalizzata: `GlobalAveragePooling2D` → `Dropout` (tasso 0.2–0.5) → `Dense` finale con Softmax sul numero di classi

In [5]:
def build_model(input_shape, num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    return model

model = build_model(INPUT_SHAPE, NUM_CLASSES)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step


## 4. Ispezione del modello

`model.summary()` per mostrare la differenza tra parametri totali e parametri addestrabili.

In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

#Total params: 23,597,957 (90.02 MB)
#Trainable params: 10,245 (40.02 KB)
#Non-trainable params: 23,587,712 (89.98 MB)

#viene addestrato solo lo 0.04% dei parametri

## 5. Compilazione e Addestramento

1. Ottimizzatore Adam e loss `categorical_crossentropy`
2. Addestramento di prova (2–5 epoche)

In [8]:
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy', #per OHE
              metrics=['accuracy']
              )

history = model.fit(X_train, y_train,
                    epochs=5,
                    batch_size=BATCH_SIZE,
                    validation_split=0.2)

model.evaluate(X_test, y_test)

Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 915ms/step - accuracy: 0.2375 - loss: 1.8821 - val_accuracy: 0.1500 - val_loss: 1.6409
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 710ms/step - accuracy: 0.1688 - loss: 1.7953 - val_accuracy: 0.2500 - val_loss: 1.5722
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 653ms/step - accuracy: 0.2125 - loss: 1.8039 - val_accuracy: 0.2500 - val_loss: 1.6056
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 668ms/step - accuracy: 0.2812 - loss: 1.6976 - val_accuracy: 0.1500 - val_loss: 1.6407
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 651ms/step - accuracy: 0.2000 - loss: 1.7133 - val_accuracy: 0.1500 - val_loss: 1.6545
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 284ms/step - accuracy: 0.1200 - loss: 1.6719


[1.671895146369934, 0.11999999731779099]